# The Environmental Impact of Cyberattacks: A Study of System Resource Utilization and Energy Consumption

(French) L’impact environnemental des cyberattaques : Étude de la sollicitation des ressources système et de la consommation énergétique

## Imports

In [ ]:
import csv
import re
from pathlib import Path
from typing import Literal, TypedDict, cast

import chardet
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup, Tag
from tqdm import tqdm

## Constants

### Numbers

In [ ]:
BYTES_PER_KB: float = 1024.0

DECIMAL_PLACES_KB: int = 2
DEFAULT_RELATIVE_TIME_OFFSET: int = 1
DEFAULT_TIME_INTERVAL_SECONDS: int = 2

ENCODING_SAMPLE_SIZE_BYTES: int = 4096

SECONDS_PER_MINUTE: int = 60

TIME_PARTS_START_INDEX: int = 3
TIMESTAMP_LENGTH_THREE_FIELDS: int = 8
TIMESTAMP_LENGTH_TWO_FIELDS: int = 5

### Paths

In [ ]:
CLEANED_DATA_DIR: Path = Path("data/cleaned")
FEATURED_DATA_DIR: Path = Path("data/featured")
RAW_DATA_DIR: Path = Path("data/raw")

### Strings

In [ ]:
CHARDET_ENCODING_KEY: str = "encoding"

DEFAULT_ENCODING: str = "utf-8"
DEFAULT_ENCODING_SIG: str = "utf-8-sig"

ENCODING_ERROR_STRATEGY: str = "replace"
EXTENSION_CSV: str = ".csv"
EXTENSION_HTML: str = ".html"
EXTENSION_LOG: str = ".log"

FILE_MODE_READ_BINARY: str = "rb"
FILE_MODE_READ_TEXT: str = "r"

GOOGLE_VISUALIZATION_ARRAY: str = (
    r"var\s+(data_\w+)\s*=\s*google\.visualization\.arrayToDataTable\(\s*\[(.*?)\]\s*\)"
)
GOOGLE_VISUALIZATION_ARRAY_HEADER: str = r"\[\s*(?:\{.*?\}\s*,\s*)?(.*?)\]"
GOOGLE_VISUALIZATION_ARRAY_ROW: str = r"\['Date\((.*?)\)'\s*,\s*(.*?)\]"

NO_EXTENSION_LABEL: str = "no_ext"

TIMESTAMP_DEFAULT_MS: str = ".000"
TIMESTAMP_FORMAT: str = "%H:%M:%S.%f"

### Tuples

In [ ]:
ATTACK_MODES: tuple[str, ...] = ("Idle", "Attack", "Recovery")
DATA_CLEANING_REPLACEMENTS: tuple[str, ...] = ("%", "C", "W", "ﺍ", "�")
SKIP_HTML_DATASETS: tuple[str, ...] = ("data_CPU_USE", "data_TOPDISK")

## Definitions

In [ ]:
class File(TypedDict, total=False):
    path: Path
    relative_path: Path
    filename: str
    extension: str
    encoding: str
    size_kb: float
    content: str
    error: str

## Dictionaries

In [ ]:
ACS712_VOLTAGES_DICT: dict[str, int] = {"hosafe": 12, "rpi4b4": 5}
"""Dictionary of ACS712 voltages for different devices."""

In [ ]:
ATTACK_CONFIGURATIONS_DICT: dict[str, dict[str, str] | int] = {
    "cryptojacking_coinimp-0%_zenbook_openhardwaremonitor": {"idle": "09:22:00.000"},
    "cryptojacking_coinimp-10%_zenbook_openhardwaremonitor": {"attack": "03:52:00.000"},
    "cryptojacking_coinimp-100%_macbook_kuman": {
        "idle": "01:00:00.000",
        "idle_browser_open": "01:03:00.000",
        "attack": "01:05:00.000",
        "recovery": "01:10:00.000",
    },
    "cryptojacking_coinimp-100%_macbook_system": {
        "idle_buffer": "00:59:34.342",
        "idle": "01:00:00.000",
        "idle_browser_open": "01:03:00.000",
        "attack": "01:05:00.000",
        "recovery": "01:10:00.000",
    },
    "cryptojacking_coinimp-100%_rpi3b_acs712": {
        "idle": "02:45:00.000",
        "idle_browser_open": "02:48:00.000",
        "attack": "02:50:00.000",
        "recovery": "02:55:00.000",
    },
    "cryptojacking_coinimp-100%_rpi3b_system": {
        "idle": "02:45:00.000",
        "idle_browser_open": "02:48:00.000",
        "attack": "02:50:00.000",
        "recovery": "02:55:00.000",
    },
    "cryptojacking_coinimp-100%_zenbook_openhardwaremonitor": {
        "attack": "09:42:00.000"
    },
    "cryptojacking_coinimp-20%_zenbook_openhardwaremonitor": {"attack": "03:30:00.000"},
    "cryptojacking_coinimp-30%_zenbook_openhardwaremonitor": {"attack": "03:01:00.000"},
    "cryptojacking_coinimp-40%_zenbook_openhardwaremonitor": {"attack": "05:26:00.000"},
    "cryptojacking_coinimp-50%_zenbook_openhardwaremonitor": {"attack": "04:52:00.000"},
    "cryptojacking_coinimp_m1_system": 1,
    "cryptojacking_gminer_victus_openhardwaremonitor": {
        "idle": "11:51:00.000",
        "attack": "11:56:00.000",
        "recovery": "12:04:00.000",
    },
    "cryptojacking_lolminer_victus_openhardwaremonitor": {
        "idle": "09:35:00.000",
        "attack": "09:40:00.000",
        "recovery": "09:46:00.000",
    },
    "cryptojacking_miniz_victus_openhardwaremonitor": {
        "idle": "16:02:00.000",
        "attack": "16:09:00.000",
        "recovery": "16:17:00.000",
    },
    "cryptojacking_nbminer_victus_openhardwaremonitor": {
        "idle": "08:00:00.000",
        "attack": "08:04:00.000",
        "recovery": "08:10:00.000",
    },
    "cryptojacking_nicehash_victus_openhardwaremonitor": {
        "idle": "19:48:00.000",
        "attack": "19:54:00.000",
        "recovery": "19:59:00.000",
    },
    "cryptojacking_onezerominer_victus_openhardwaremonitor": {
        "idle": "00:21:28.000",
        "attack": "00:28:58.000",
        "recovery": "00:34:56.000",
    },
    "cryptojacking_srbminer_victus_openhardwaremonitor": {
        "idle": "20:20:00.000",
        "attack": "20:25:00.000",
        "recovery": "20:30:00.000",
    },
    "cryptojacking_t-rex_msi_system": 1,
    "cryptojacking_t-rex_zenbook_openhardwaremonitor": {
        "idle": "11:06:00.000",
        "attack": "11:10:42.000",
    },
    "cryptojacking_wildrig_victus_openhardwaremonitor": {
        "idle": "11:55:00.000",
        "attack": "12:00:00.000",
        "recovery": "12:06:00.000",
    },
    "cryptojacking_xmrig_rpi3b_system": 1,
    "cryptojacking_xmrig_rpi4b2_system": 1,
    "cryptojacking_xmrig_rpi4b4_system_01": 1,
    "cryptojacking_xmrig_rpi4b4_system_02": 1,
    "cryptojacking_xmrig_rpi4b4_system_03": 1,
    "cryptojacking_xmrig_rpi5_system": 1,
    "denial-of-service_goldeneye_rpi4b4_acs712": 200,
    "denial-of-service_goldeneye_rpi4b4_nmon": 200,
    "denial-of-service_goldeneye_rpi4b4_system": 200,
    "denial-of-service_goloris_hosafe_acs712": 200,
    "denial-of-service_goloris_rpi4b4_acs712": 200,
    "denial-of-service_goloris_rpi4b4_nmon": 200,
    "denial-of-service_hping3_d-link_kuman": 1,
    "denial-of-service_hping3_hosafe_acs712": 1,
    "denial-of-service_hping3_huawei_acs712": 1,
    "denial-of-service_hping3_tp-link_acs712": 1,
    "denial-of-service_hulk_rpi4b4_acs712": 200,
    "denial-of-service_hulk_rpi4b4_nmon": 200,
    "denial-of-service_hulk_rpi4b4_system": 200,
    "denial-of-service_mhddos-icmp_hosafe_acs712": 200,
    "denial-of-service_mhddos-icmp_rpi4b4_acs712": 200,
    "denial-of-service_mhddos-icmp_rpi4b4_nmon": 200,
    "denial-of-service_mhddos-tcp_hosafe_acs712": 200,
    "denial-of-service_mhddos-tcp_rpi4b4_acs712": 200,
    "denial-of-service_mhddos-tcp_rpi4b4_nmon": 200,
    "denial-of-service_mhddos-udp_hosafe_acs712": 200,
    "denial-of-service_mhddos-udp_rpi4b4_acs712": 200,
    "denial-of-service_mhddos-udp_rpi4b4_nmon": 200,
    "denial-of-service_pyflooder_rpi4b8_system": {
        "idle": "03:32:00.000",
        "attack": "03:35:00.000",
        "recovery": "03:57:10.000",
    },
    "denial-of-service_slowloris_rpi4b2_acs712_01": {
        "idle": "14:50:27.000",
        "attack": "14:52:28.000",
        "recovery": "14:54:28.000",
    },
    "denial-of-service_slowloris_rpi4b2_acs712_02": {
        "idle": "15:01:45.000",
        "attack": "15:03:46.000",
        "recovery": "15:05:46.000",
    },
    "denial-of-service_slowloris_rpi4b2_acs712_03": {
        "idle": "15:09:46.000",
        "attack": "15:11:47.000",
        "recovery": "15:13:47.000",
    },
    "denial-of-service_slowloris_rpi4b2_system_01": {
        "idle": "14:50:26.000",
        "attack": "14:52:26.000",
        "recovery": "14:54:26.000",
    },
    "denial-of-service_slowloris_rpi4b2_system_02": {
        "idle": "15:01:43.000",
        "attack": "15:03:44.000",
        "recovery": "15:05:44.000",
    },
    "denial-of-service_slowloris_rpi4b2_system_03": {
        "idle": "15:09:45.000",
        "attack": "15:11:45.000",
        "recovery": "15:13:45.000",
    },
    "ransomware_bstry_rpi4b2_acs712": {
        "idle": "21:13:25.000",
        "attack": "21:14:26.000",
        "recovery": "21:14:36.000",
    },
    "ransomware_bstry_rpi4b2_system": {
        "idle": "21:13:25.000",
        "attack": "21:14:25.000",
        "recovery": "21:14:35.000",
    },
    "ransomware_jigsaw_rpi4b4_prometheus": {
        "idle": "00:00:00.000",
        "attack": "00:05:00.000",
        "recovery": "00:10:00.000",
    },
    "ransomware_jigsaw_vm_prometheus": {
        "idle": "00:00:00.000",
        "attack": "00:05:00.000",
        "recovery": "00:10:00.000",
    },
    "ransomware_petya_rpi4b4_prometheus": {
        "idle": "00:00:00.000",
        "attack": "00:05:00.000",
        "recovery": "00:10:00.000",
    },
    "ransomware_petya_vm_prometheus": {
        "idle": "00:00:00.000",
        "attack": "00:05:00.000",
        "recovery": "00:10:00.000",
    },
    "ransomware_randomware_rpi4b2_acs712": {
        "idle": "19:55:37.000",
        "attack": "19:57:38.000",
        "recovery": "19:57:48.000",
    },
    "ransomware_randomware_rpi4b2_system": {
        "idle": "19:55:37.000",
        "attack": "19:57:37.000",
        "recovery": "19:57:47.000",
    },
    "ransomware_ransomware-poc_rpi4b2_acs712_01": {
        "idle": "19:55:44.000",
        "attack": "19:56:44.000",
        "recovery": "19:58:50.000",
    },
    "ransomware_ransomware-poc_rpi4b2_acs712_02": {
        "idle": "21:14:12.000",
        "attack": "21:16:13.000",
        "recovery": "21:18:13.000",
    },
    "ransomware_ransomware-poc_rpi4b2_system_01": {
        "idle": "19:55:44.000",
        "attack": "19:56:44.000",
        "recovery": "19:58:49.000",
    },
    "ransomware_ransomware-poc_rpi4b2_system_02": {
        "idle": "21:14:14.000",
        "attack": "21:16:14.000",
        "recovery": "21:18:14.000",
    },
    "ransomware_rex_rpi4b4_prometheus": {
        "idle": "00:00:00.000",
        "attack": "00:05:00.000",
        "recovery": "00:10:00.000",
    },
    "ransomware_rex_vm_prometheus": {
        "idle": "00:00:00.000",
        "attack": "00:05:00.000",
        "recovery": "00:10:00.000",
    },
    "ransomware_thanos_rpi4b4_prometheus": {
        "idle": "00:00:00.000",
        "attack": "00:05:00.000",
        "recovery": "00:10:00.000",
    },
    "ransomware_thanos_vm_prometheus": {
        "idle": "00:00:00.000",
        "attack": "00:05:00.000",
        "recovery": "00:10:00.000",
    },
    "ransomware_wannacry_rpi4b4_prometheus": {
        "idle": "00:00:00.000",
        "attack": "00:05:00.000",
        "recovery": "00:10:00.000",
    },
    "ransomware_wannacry_vm_prometheus": {
        "idle": "00:00:00.000",
        "attack": "00:05:00.000",
        "recovery": "00:10:00.000",
    },
}

In [ ]:
ATTACK_TYPES_DICT: dict[str, str] = {
    "cryptojacking": "Cryptojacking",
    "denial-of-service": "Denial of Service",
    "ransomware": "Ransomware",
}

In [ ]:
ATTACKS_DICT: dict[str, str] = {
    "bstry": "Bstry",
    "coinimp": "CoinIMP (CPU) (100%)",
    "coinimp-0%": "CoinIMP (CPU) (0%)",
    "coinimp-10%": "CoinIMP (CPU) (10%)",
    "coinimp-100%": "CoinIMP (CPU) (100%)",
    "coinimp-20%": "CoinIMP (CPU) (20%)",
    "coinimp-30%": "CoinIMP (CPU) (30%)",
    "coinimp-40%": "CoinIMP (CPU) (40%)",
    "coinimp-50%": "CoinIMP (CPU) (50%)",
    "gminer": "GMiner (GPU)",
    "goldeneye": "GoldenEye (HTTP)",
    "goloris": "Goloris (HTTP)",
    "hping3": "Hping3 (SYN)",
    "hulk": "HULK (HTTP)",
    "jigsaw": "Jigsaw",
    "lolminer": "lolMiner (GPU)",
    "mhddos-icmp": "MHDDoS (ICMP)",
    "mhddos-tcp": "MHDDoS (TCP)",
    "mhddos-udp": "MHDDoS (UDP)",
    "miniz": "miniZ (GPU)",
    "nbminer": "NBMiner (GPU)",
    "nicehash": "NiceHash (GPU)",
    "onezerominer": "OneZeroMiner (GPU)",
    "petya": "Petya",
    "pyflooder": "PyFlooder (HTTP)",
    "randomware": "Randomware",
    "ransomware-poc": "Ransomware-PoC",
    "rex": "Rex",
    "slowloris": "Slowloris (HTTP)",
    "srbminer": "SRBMiner Multi (GPU)",
    "t-rex": "T-Rex (GPU)",
    "thanos": "Thanos",
    "wannacry": "WannaCry",
    "wildrig": "WildRig Multi (GPU)",
    "xmrig": "XMRig (CPU)",
}

In [ ]:
CLEAN_HEADERS_DICT: dict[str, str] = {
    "CPU (%)": "CPU Usage (%)",
    "CPU Core #1": "CPU Temperature Core #1 (°C)",
    "CPU Core #1 Temp (°C)": "CPU Temperature Core #1 (°C)",
    "CPU Core #2": "CPU Temperature Core #2 (°C)",
    "CPU Core #2 Temp (°C)": "CPU Temperature Core #2 (°C)",
    "CPU Core #3": "CPU Temperature Core #3 (°C)",
    "CPU Core #3 Temp (°C)": "CPU Temperature Core #3 (°C)",
    "CPU Core #4": "CPU Temperature Core #4 (°C)",
    "CPU Core #4 Temp (°C)": "CPU Temperature Core #4 (°C)",
    "CPU Cores Power": "CPU Power Cores (W)",
    "CPU Cores Power (W)": "CPU Power Cores (W)",
    "CPU Graphics Power": "CPU Graphics Power (W)",
    "CPU Graphics Power (W)": "CPU Graphics Power (W)",
    "CPU Package": "CPU Temperature (°C)",
    "CPU Package Power": "CPU Power (W)",
    "CPU Package Power (W)": "CPU Power (W)",
    "CPU Package Temp (°C)": "CPU Temperature (°C)",
    "CPU Temperature (m°C)": "CPU Temperature (m°C)",
    "CPU Temperature (°C)": "CPU Temperature (°C)",
    "CPU Total Load (%)": "CPU Usage (%)",
    "CPU Usage (%)": "CPU Usage (%)",
    "Consommation (W)": "Power (W)",
    "Courant (A)": "Current (A)",
    "Current (A)": "Current (A)",
    "Current (mA)": "Current (mA)",
    "Disque chiffré (%)": "Encrypted Disk Usage (%)",
    "Ecriture disque (Mo/s)": "Disk Write (Mo/s)",
    "Fichiers chiffrés": "Encrypted Files",
    "GPU Core": "GPU Temperature (°C)",
    "GPU Core Load (%)": "GPU Usage (%)",
    "GPU Core Temp (°C)": "GPU Temperature (°C)",
    "GPU Memory Load (%)": "GPU Memory Usage (%)",
    "GPU Power Consumption (W)": "GPU Power (W)",
    "GPU Usage (%)": "GPU Usage (%)",
    "Horodatage": "Timestamp",
    "Lecture disque (Mo/s)": "Disk Read (Mo/s)",
    "Memory Usage (%)": "Memory Usage (%)",
    "Power (W)": "Power (W)",
    "Power Consumption (W)": "Power (W)",
    "Puissance (W)": "Power (W)",
    "RAM (%)": "Memory Usage (%)",
    "RAM (Mo)": "Memory Usage (Mo)",
    "RAM Load (%)": "Memory Usage (%)",
    "RAM Usage (%)": "Memory Usage (%)",
    "Réseau (Ko/s)": "Network (Ko/s)",
    "Taille fichiers chiffres (Mo)": "Encrypted Files Size (Mo)",
    "Temps": "Timestamp",
    "Temps relatif (s)": "Relative Time (s)",
    "Température (°C)": "CPU Temperature (°C)",
    "Tension (V)": "Voltage (V)",
    "Timestamp": "Timestamp",
    "Timestamp (s)": "Relative Time (s)",
    "Vitesse chiffrement (o/s)": "Encryption Speed (o/s)",
    "Voltage (V)": "Voltage (V)",
    "Voltage Adjusted (V)": "Voltage Adjusted (V)",
    "data_CPU_UTIL_Idle%": "CPU Usage Idle (%)",
    "data_CPU_UTIL_Sys%": "CPU Usage System (%)",
    "data_CPU_UTIL_User%": "CPU Usage User (%)",
    "data_CPU_UTIL_Wait%": "CPU Usage Wait (%)",
    "data_DISKBSIZE_mmcblk0": "Disk mmcblk0 Block Size (Ko)",
    "data_DISKBSIZE_mmcblk0p1": "Disk mmcblk0p1 Block Size (Ko)",
    "data_DISKBSIZE_mmcblk0p2": "Disk mmcblk0p2 Block Size (Ko)",
    "data_DISKBSIZE_mmcblk1": "Disk mmcblk1 Block Size (Ko)",
    "data_DISKBSIZE_mmcblk1p1": "Disk mmcblk1p1 Block Size (Ko)",
    "data_DISKBSIZE_mmcblk1p2": "Disk mmcblk1p2 Block Size (Ko)",
    "data_DISKBUSY_mmcblk0": "Disk mmcblk0 Usage (%)",
    "data_DISKBUSY_mmcblk0p1": "Disk mmcblk0p1 Usage (%)",
    "data_DISKBUSY_mmcblk0p2": "Disk mmcblk0p2 Usage (%)",
    "data_DISKBUSY_mmcblk1": "Disk mmcblk1 Usage (%)",
    "data_DISKBUSY_mmcblk1p1": "Disk mmcblk1p1 Usage (%)",
    "data_DISKBUSY_mmcblk1p2": "Disk mmcblk1p2 Usage (%)",
    "data_DISKREAD_mmcblk0": "Disk mmcblk0 Read (Ko/s)",
    "data_DISKREAD_mmcblk0p1": "Disk mmcblk0p1 Read (Ko/s)",
    "data_DISKREAD_mmcblk0p2": "Disk mmcblk0p2 Read (Ko/s)",
    "data_DISKREAD_mmcblk1": "Disk mmcblk1 Read (Ko/s)",
    "data_DISKREAD_mmcblk1p1": "Disk mmcblk1p1 Read (Ko/s)",
    "data_DISKREAD_mmcblk1p2": "Disk mmcblk1p2 Read (Ko/s)",
    "data_DISKWRITE_mmcblk0": "Disk mmcblk0 Write (Ko/s)",
    "data_DISKWRITE_mmcblk0p1": "Disk mmcblk0p1 Write (Ko/s)",
    "data_DISKWRITE_mmcblk0p2": "Disk mmcblk0p2 Write (Ko/s)",
    "data_DISKWRITE_mmcblk1": "Disk mmcblk1 Write (Ko/s)",
    "data_DISKWRITE_mmcblk1p1": "Disk mmcblk1p1 Write (Ko/s)",
    "data_DISKWRITE_mmcblk1p2": "Disk mmcblk1p2 Write (Ko/s)",
    "data_DISKXFER_mmcblk0": "Disk mmcblk0 Transfers",
    "data_DISKXFER_mmcblk0p1": "Disk mmcblk0p1 Transfers",
    "data_DISKXFER_mmcblk0p2": "Disk mmcblk0p2 Transfers",
    "data_DISKXFER_mmcblk1": "Disk mmcblk1 Transfers",
    "data_DISKXFER_mmcblk1p1": "Disk mmcblk1p1 Transfers",
    "data_DISKXFER_mmcblk1p2": "Disk mmcblk1p2 Transfers",
    "data_FORKEXEC_exec": "Process Execs",
    "data_FORKEXEC_fork": "Process Forks",
    "data_JFS_/": "JFS Usage (%)",
    "data_JFS_/boot/firmware": "JFS boot/firmware Usage (%)",
    "data_JFS_/dev": "JFS dev Usage (%)",
    "data_JFS_/run": "JFS run Usage (%)",
    "data_MEM_LINUX_active": "Memory Active (Mo)",
    "data_MEM_LINUX_buffers": "Memory Buffers (Mo)",
    "data_MEM_LINUX_cached": "Memory Cached (Mo)",
    "data_MEM_LINUX_inactive": "Memory Inactive (Mo)",
    "data_MEM_LINUX_memfree": "Memory Free (Mo)",
    "data_MEM_LINUX_memtotal": "Memory Total (Mo)",
    "data_NETPACKET_eth0-read/s": "Network Packet eth0 Received",
    "data_NETPACKET_eth0-write/s": "Network Packet eth0 Sent",
    "data_NETPACKET_lo-read/s": "Network Packet lo Received",
    "data_NETPACKET_lo-write/s": "Network Packet lo Sent",
    "data_NETPACKET_wlan0-read/s": "Network Packet wlan0 Received",
    "data_NETPACKET_wlan0-write/s": "Network Packet wlan0 Sent",
    "data_NET_eth0-read-KB/s": "Network eth0 Received (Ko/s)",
    "data_NET_eth0-write-KB/s": "Network eth0 Sent (Ko/s)",
    "data_NET_lo-read-KB/s": "Network lo Received (Ko/s)",
    "data_NET_lo-write-KB/s": "Network lo Sent (Ko/s)",
    "data_NET_wlan0-read-KB/s": "Network wlan0 Received (Ko/s)",
    "data_NET_wlan0-write-KB/s": "Network wlan0 Sent (Ko/s)",
    "data_PSWITCH_pswitch": "Process Switches",
    "data_RUNQBLOCK_Blocked": "Processes Blocked",
    "data_RUNQBLOCK_Runnable": "Processes Runnable",
    "data_SWAP_LINUX_swapfree": "Swap Free (Mo)",
    "data_SWAP_LINUX_swaptotal": "Swap Total (Mo)",
}
"""Dictionary of headers to rename."""

In [ ]:
HOSTS_DICT: dict[str, str] = {
    "d-link": "D-Link DIR-822",
    "hosafe": "HOSAFE HX-2PT1",
    "huawei": "Huawei H151-381",
    "m1": "Apple M1 Pro",
    "macbook": "Apple MacBook Pro i7 2013",
    "msi": "Windows 11 i7 GeForce RTX 4060 Laptop",
    "rpi3b": "Raspberry Pi 3B 1GB",
    "rpi4b2": "Raspberry Pi 4B 2GB",
    "rpi4b4": "Raspberry Pi 4B 4GB",
    "rpi4b8": "Raspberry Pi 4B 8GB",
    "rpi5": "Raspberry Pi 5",
    "tp-link": "TP-Link Tapo C200",
    "victus": "HP Victus 16-d0417nf",
    "vm": "Windows 10 VM",
    "zenbook": "ASUS Zenbook UX51VZ",
}

In [ ]:
MEASUREMENT_TOOLS_DICT: dict[str, str] = {
    "acs712": "ACS712",
    "kuman": "Kuman KW47",
    "nmon": "nmon",
    "openhardwaremonitor": "Open Hardware Monitor",
    "prometheus": "Prometheus",
    "system": "System",
}

In [ ]:
MISSING_HEADERS_DICT: dict[str, list[str]] = {
    "cryptojacking_coinimp-100%_rpi3b_acs712.csv": [
        "Timestamp",
        "Voltage (V)",
        "Voltage Adjusted (V)",
        "Current (A)",
        "Power (W)",
    ],
    "denial-of-service_goldeneye_rpi4b4_acs712.log": ["Current (mA)"],
    "denial-of-service_goldeneye_rpi4b4_system.log": ["CPU Temperature (m°C)"],
    "denial-of-service_goloris_hosafe_acs712.log": ["Current (mA)"],
    "denial-of-service_goloris_rpi4b4_acs712.log": ["Current (mA)"],
    "denial-of-service_hulk_rpi4b4_acs712.log": ["Current (mA)"],
    "denial-of-service_hulk_rpi4b4_system.log": ["CPU Temperature (m°C)"],
    "denial-of-service_mhddos-icmp_hosafe_acs712.log": ["Current (mA)"],
    "denial-of-service_mhddos-icmp_rpi4b4_acs712.log": ["Current (mA)"],
    "denial-of-service_mhddos-tcp_hosafe_acs712.log": ["Current (mA)"],
    "denial-of-service_mhddos-tcp_rpi4b4_acs712.log": ["Current (mA)"],
    "denial-of-service_mhddos-udp_hosafe_acs712.log": ["Current (mA)"],
    "denial-of-service_mhddos-udp_rpi4b4_acs712.log": ["Current (mA)"],
}
"""Dictionary of missing headers for each dataset."""

## Regex

In [ ]:
GOOGLE_VISUALIZATION_ARRAY_PATTERN = re.compile(GOOGLE_VISUALIZATION_ARRAY, re.DOTALL)
"""Pattern to match a Google Visualization array."""

GOOGLE_VISUALIZATION_ARRAY_HEADER_PATTERN = re.compile(
    GOOGLE_VISUALIZATION_ARRAY_HEADER, re.DOTALL
)
"""Pattern to match a Google Visualization array header."""

GOOGLE_VISUALIZATION_ARRAY_ROW_PATTERN = re.compile(
    GOOGLE_VISUALIZATION_ARRAY_ROW, re.DOTALL
)
"""Pattern to match a Google Visualization array row."""

## Functions

### Standalone Functions

In [ ]:
def add_attack_metadata(df: pd.DataFrame, filename: str) -> pd.DataFrame:
    """
    Add attack metadata to a dataframe.

    Args:
        df (pd.DataFrame): The dataframe to add attack metadata to.
        filename (str): The filename of the dataset.

    Returns:
        pd.DataFrame: The dataframe with attack metadata added.
    """

    elements: list[str] = filename.split("_")

    df["Attack Type"] = ATTACK_TYPES_DICT.get(elements[0], elements[0])
    df["Attack"] = ATTACKS_DICT.get(elements[1], elements[1])
    df["Host"] = HOSTS_DICT.get(elements[2], elements[2])
    df["Measurement"] = MEASUREMENT_TOOLS_DICT.get(
        elements[3].split(".")[0], elements[3].split(".")[0]
    )
    df["Run"] = int(elements[4].split(".")[0]) if len(elements) == 5 else 1

    return df

In [ ]:
def add_relative_time(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add relative time to a dataframe.

    Args:
        df (pd.DataFrame): The dataframe to add relative time to.

    Returns:
        pd.DataFrame: The dataframe with relative time added.
    """

    if "Relative Time (s)" not in df.columns:
        if "Timestamp" in df.columns:
            timestamps = pd.to_datetime(df["Timestamp"], format=TIMESTAMP_FORMAT)
            start_time = timestamps.min()

            df["Relative Time (s)"] = (timestamps - start_time).dt.total_seconds()
        else:
            df["Relative Time (s)"] = np.arange(len(df)).astype(float)

    return df

In [ ]:
def assign_attack_mode(
    df: pd.DataFrame, configuration: dict[str, str] | int
) -> pd.DataFrame:
    """
    Assign attack mode to a dataframe.

    Args:
        df (pd.DataFrame): The dataframe to assign attack mode to.
        configuration (dict[str, str] | int): The attack mode configuration.

    Returns:
        pd.DataFrame: The dataframe with attack mode assigned.
    """

    df["Mode"] = ""
    configuration_type: str = type(configuration).__name__

    if configuration_type == "dict":
        for mode, timestamp in cast(dict[str, str], configuration).items():
            df["Mode"] = np.where(
                timestamp <= df["Timestamp"], mode.capitalize(), df["Mode"]
            )
    elif configuration_type == "int":
        configuration = cast(int, configuration)
        rows_count: int = df.shape[0]
        modes: int = round(rows_count / configuration)

        if modes == 2:
            df["Mode"] = np.where(
                df.index < configuration, ATTACK_MODES[0], ATTACK_MODES[1]
            )
        elif modes == 3:
            conditions: list[np.ndarray] = [
                df.index < configuration,
                (df.index >= configuration) & (df.index < 2 * configuration),
                df.index >= 2 * configuration,
            ]
            df["Mode"] = np.select(conditions, ATTACK_MODES, default="")

    return df

In [ ]:
def calculate_cpu_usage(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate CPU usage in a dataframe.

    Args:
        df (pd.DataFrame): The dataframe to calculate CPU usage for.

    Returns:
        pd.DataFrame: The dataframe with CPU usage calculated.
    """

    if "CPU Usage Idle (%)" in df.columns:
        df["CPU Usage (%)"] = 100 - df["CPU Usage Idle (%)"]
        df.drop(columns=["CPU Usage Idle (%)"], inplace=True)

    return df

In [ ]:
def calculate_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate memory usage in a dataframe.

    Args:
        df (pd.DataFrame): The dataframe to calculate memory usage for.

    Returns:
        pd.DataFrame: The dataframe with memory usage calculated.
    """

    if "Memory Free (Mo)" in df.columns and "Memory Total (Mo)" in df.columns:
        df["Memory Usage (%)"] = (
            1 - df["Memory Free (Mo)"] / df["Memory Total (Mo)"]
        ) * 100
        df["Memory Active (%)"] = (
            df["Memory Active (Mo)"] / df["Memory Total (Mo)"]
        ) * 100
        df["Memory Buffers (%)"] = (
            df["Memory Buffers (Mo)"] / df["Memory Total (Mo)"]
        ) * 100
        df["Memory Cached (%)"] = (
            df["Memory Cached (Mo)"] / df["Memory Total (Mo)"]
        ) * 100
        df["Memory Inactive (%)"] = (
            df["Memory Inactive (Mo)"] / df["Memory Total (Mo)"]
        ) * 100
        df.drop(
            columns=[
                "Memory Free (Mo)",
                "Memory Active (Mo)",
                "Memory Buffers (Mo)",
                "Memory Cached (Mo)",
                "Memory Inactive (Mo)",
                "Memory Total (Mo)",
            ],
            inplace=True,
        )

    return df

In [ ]:
def clean_data(data: pd.Series) -> pd.Series:
    """
    Clean data from the dataset.

    Args:
        data (pd.Series): The data to clean.

    Returns:
        pd.Series: The cleaned data.
    """

    pattern: str = "|".join(map(re.escape, DATA_CLEANING_REPLACEMENTS))
    data = (
        data.astype(str)
        .str.replace(",", ".")
        .str.replace(pattern, "", regex=True)
        .str.rstrip()
    )

    return pd.to_numeric(data, errors="coerce")

In [ ]:
def clean_memory_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean memory columns from the dataset.

    Args:
        df (pd.DataFrame): The dataframe to clean.

    Returns:
        pd.DataFrame: The cleaned dataframe.
    """

    if "Memory Usage (Mo)" in df.columns and "Memory Usage (%)" in df.columns:
        df.drop(columns=["Memory Usage (Mo)"], inplace=True)

    return df

In [ ]:
def clean_timestamp(
    timestamps: pd.Series, relative_time: pd.Series | None = None
) -> pd.Series:
    """
    Clean timestamps from the dataset.

    Args:
        timestamps (pd.Series): The timestamps to clean.
        relative_time (pd.Series | None, optional): The relative time to add to the timestamps.

    Returns:
        pd.Series: The cleaned timestamps.
    """

    timestamps = (
        timestamps.str.split(" ").str[-1].str.split("–").str[0].str.replace("/", ":")
    )

    first_timestamp: str = timestamps.iloc[0]
    last_timestamp: str = timestamps.iloc[-1]

    timestamp_length: int = len(first_timestamp)

    if timestamp_length == TIMESTAMP_LENGTH_THREE_FIELDS:
        timestamps = timestamps + TIMESTAMP_DEFAULT_MS
    elif timestamp_length == TIMESTAMP_LENGTH_TWO_FIELDS:
        left_is_identical: bool = first_timestamp[:2] == last_timestamp[:2]
        right_is_identical: bool = first_timestamp[3:] == last_timestamp[3:]

        if left_is_identical and not right_is_identical:
            relative_time_offset: int = DEFAULT_RELATIVE_TIME_OFFSET
            delta: pd.Series | pd.TimedeltaIndex

            if relative_time is not None:
                delta = pd.to_timedelta(relative_time, unit="s")
            else:
                delta = pd.to_timedelta(
                    range(
                        0,
                        len(timestamps) * DEFAULT_TIME_INTERVAL_SECONDS,
                        DEFAULT_TIME_INTERVAL_SECONDS,
                    ),
                    unit="s",
                )
                relative_time_offset = 2

            offset: pd.Timedelta = pd.to_timedelta(
                SECONDS_PER_MINUTE
                - (timestamps == first_timestamp).sum() * relative_time_offset,
                unit="s",
            )

            timestamps = pd.Series(
                pd.to_datetime(first_timestamp, format="%H:%M") + delta + offset
            )
            timestamps = timestamps.dt.strftime(TIMESTAMP_FORMAT).str[:-3]
        elif not left_is_identical and right_is_identical:
            timestamps = "00:" + timestamps + TIMESTAMP_DEFAULT_MS

    return timestamps

In [ ]:
def convert_current_units(df: pd.DataFrame, filename: str) -> pd.DataFrame:
    """
    Convert current units in a dataframe.

    Args:
        df (pd.DataFrame): The dataframe to convert.

    Returns:
        pd.DataFrame: The converted dataframe.
    """

    if "Current (mA)" in df.columns:
        df["Current (A)"] = df["Current (mA)"] / 1000
        df.drop(columns=["Current (mA)"], inplace=True)

        voltage: int = ACS712_VOLTAGES_DICT.get(filename.split("_")[2], 1)
        df["Power (W)"] = df["Current (A)"] * voltage

    return df

In [ ]:
def convert_temperature_units(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert temperature units in a dataframe.

    Args:
        df (pd.DataFrame): The dataframe to convert.

    Returns:
        pd.DataFrame: The converted dataframe.
    """

    if "CPU Temperature (m°C)" in df.columns:
        df["CPU Temperature (°C)"] = df["CPU Temperature (m°C)"] / 1000
        df.drop(columns=["CPU Temperature (m°C)"], inplace=True)

    return df

In [ ]:
def detect_encoding(
    file_path: Path, sample_size: int = ENCODING_SAMPLE_SIZE_BYTES
) -> str:
    """
    Detect the encoding of a file.

    Args:
        file_path (Path): The path to the file.
        sample_size (int): The number of bytes to sample from the file.

    Returns:
        str: The detected encoding.
    """

    try:
        with open(file_path, FILE_MODE_READ_BINARY) as f:
            sample_bytes: bytes = f.read(sample_size)

        return chardet.detect(sample_bytes).get(CHARDET_ENCODING_KEY, DEFAULT_ENCODING)

    except (OSError, UnicodeDecodeError) as e:
        print(f"[{e}] Failed to detect encoding for {file_path}")
        return DEFAULT_ENCODING

In [ ]:
def detect_line_separator(line: str) -> str:
    """
    Detect the line separator in a string.

    Args:
        line (str): The string to detect the line separator in.

    Returns:
        str: The detected line separator.
    """

    try:
        sniffer: csv.Sniffer = csv.Sniffer()
        return sniffer.sniff(line, delimiters="\t,;|").delimiter

    except csv.Error as e:
        print(f"[{e}] Failed to detect line separator for {line}")
        return ","

In [ ]:
def get_files(directory_path: Path) -> list[Path]:
    """
    List all files in a directory.

    Args:
        directory_path (Path): The directory to list files from.

    Returns:
        list[Path]: A list of all files in the directory.
    """

    if not directory_path.exists():
        raise ValueError(f"[ValueError] {directory_path} does not exist.")

    return sorted([f for f in directory_path.rglob("*") if f.is_file()])

In [ ]:
def is_header(line: str) -> bool:
    """
    Check if a line is a header.

    Args:
        line (str): The line to check.

    Returns:
        bool: True if the line is a header, False otherwise.
    """

    if "<" in line:
        return False

    return re.search(r"[a-zA-Z]", line) is not None

In [ ]:
def merge(
    dfs: list[pd.DataFrame],
    on: str,
    how: Literal["left", "right", "outer", "inner", "cross"] = "inner",
) -> pd.DataFrame:
    """
    Merge a list of dataframes.

    Args:
        dfs (list[pd.DataFrame]): A list of dataframes to merge.
        on (str): The column to merge on.
        how (Literal["left", "right", "outer", "inner", "cross"]): The merge type.

    Returns:
        pd.DataFrame: The merged dataframe.
    """

    merged_df: pd.DataFrame = dfs[0]

    for df in dfs[1:]:
        merged_df = merged_df.merge(df, on=on, how=how)

    return merged_df

In [ ]:
def read_file_content(file_path: Path, encoding: str) -> tuple[str, str | None]:
    """
    Read the content of a file.

    Args:
        file_path (Path): The path to the file.
        encoding (str): The encoding of the file.

    Returns:
        tuple[str, str | None]: A tuple containing the content of the file and an error message if any.
    """

    try:
        with open(
            file_path,
            FILE_MODE_READ_TEXT,
            encoding=encoding,
            errors=ENCODING_ERROR_STRATEGY,
        ) as f:
            return f.read(), None

    except (OSError, UnicodeDecodeError) as e:
        print(f"[{e}] Failed to read {file_path}")
        return "", str(e)

In [ ]:
def remove_mojibake(text: str) -> str:
    """
    Remove mojibake characters from a string.

    Args:
        text (str): The string to remove mojibake characters from.

    Returns:
        str: The string without mojibake characters.
    """

    mojibake_mapping: dict[str, str] = {
        "ﺍ": "°",
        "�": "°",
    }

    return text.translate(str.maketrans(mojibake_mapping))

In [ ]:
def remove_zero_swap_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove columns with zero swap.

    Args:
        df (pd.DataFrame): The dataframe to remove columns from.

    Returns:
        pd.DataFrame: The dataframe without columns with zero swap.
    """

    if (
        "Swap Free (Mo)" in df.columns
        and "Swap Total (Mo)" in df.columns
        and (df["Swap Total (Mo)"] == 0).all()
    ):
        df.drop(columns=["Swap Free (Mo)", "Swap Total (Mo)"], inplace=True)

    return df

In [ ]:
def save_dataframe(df: pd.DataFrame, output_path: Path) -> None:
    """
    Save a dataframe to a file.

    Args:
        df (pd.DataFrame): The dataframe to save.
        output_path (Path): The path to save the dataframe to.
    """

    output_path.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(output_path, encoding=DEFAULT_ENCODING_SIG, index=False)

### Composite 1 Functions

In [ ]:
def clean_headers(headers: list[str]) -> list[str]:
    """ "
    Clean headers.

    Args:
        headers (list[str]): The headers to clean.

    Returns:
        list[str]: The cleaned headers.
    """

    headers = [remove_mojibake(header) for header in headers]

    return [CLEAN_HEADERS_DICT.get(header, header) for header in headers]

In [ ]:
def create_file_entry(file_path: Path, base_dir_path: Path) -> File:
    """
    Create a file entry.

    Args:
        file_path (Path): The path to the file.
        base_dir_path (Path): The base directory path.

    Returns:
        File: A file entry.
    """

    encoding: str = detect_encoding(file_path)
    rel_path: Path = file_path.relative_to(base_dir_path)
    size_kb: float = round(file_path.stat().st_size / BYTES_PER_KB, DECIMAL_PLACES_KB)

    content: str
    error: str | None
    content, error = read_file_content(file_path, encoding)

    entry: File = {
        "path": file_path,
        "relative_path": rel_path,
        "filename": file_path.name,
        "extension": file_path.suffix.lower() or NO_EXTENSION_LABEL,
        "encoding": encoding,
        "size_kb": size_kb,
        "content": content,
    }

    if error:
        entry["error"] = error

    return entry

In [ ]:
def feature_dataframe(df: pd.DataFrame, filename: str) -> pd.DataFrame:
    """
    Feature a dataframe.

    Args:
        df (pd.DataFrame): The dataframe to feature.

    Returns:
        pd.DataFrame: The featured dataframe.
    """

    configuration: dict[str, str] | int = ATTACK_CONFIGURATIONS_DICT.get(
        Path(filename).stem, {}
    )
    if not configuration:
        print(f"Configuration not found for {filename}")

    df = add_relative_time(df)
    df = convert_temperature_units(df)
    df = calculate_cpu_usage(df)
    df = convert_current_units(df, filename)
    df = calculate_memory_usage(df)
    df = clean_memory_columns(df)
    df = remove_zero_swap_columns(df)
    df = assign_attack_mode(df, configuration)
    df = add_attack_metadata(df, filename)

    return df

In [ ]:
def load_dataframe_from_html(markup: str) -> pd.DataFrame:
    """
    Load a dataframe from HTML markup.

    Args:
        markup (str): The HTML markup to load the dataframe from.

    Returns:
        pd.DataFrame: The loaded dataframe.
    """

    if not markup or not markup.strip():
        return pd.DataFrame()

    soup: BeautifulSoup = BeautifulSoup(markup, "html.parser")
    script: Tag = soup.find_all("script")[-1]

    script_text: str = script.get_text()
    if not script_text:
        return pd.DataFrame()

    dfs: list[pd.DataFrame] = []

    for resource, text in GOOGLE_VISUALIZATION_ARRAY_PATTERN.findall(script_text):
        if resource in SKIP_HTML_DATASETS:
            continue

        data: list[list[str]] = [
            ["Timestamp"]
            + [
                part.strip("'")
                for part in GOOGLE_VISUALIZATION_ARRAY_HEADER_PATTERN.findall(text)[
                    0
                ].split(",")
            ]
        ]

        for datetime, values in GOOGLE_VISUALIZATION_ARRAY_ROW_PATTERN.findall(text):
            time: str = ":".join(
                [part.strip() for part in datetime.split(",")[TIME_PARTS_START_INDEX:]]
            )

            data.append([time] + values.split(","))

        df: pd.DataFrame = pd.DataFrame(data[1:], columns=data[0])
        df = df.rename(
            columns={
                column: f"{resource}_{column}"
                for column in df.columns
                if column != "Timestamp"
            }
        )

        dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    return merge(dfs, "Timestamp", "outer")

### Composite 2 Functions

In [ ]:
def build_dataset(directory_path: Path) -> list[File]:
    """
    Build a dataset of files.

    Args:
        directory_path (Path): The directory to build the dataset from.

    Returns:
        list[File]: A list of files.
    """

    files: list[Path] = get_files(directory_path)
    dataset: list[File] = []

    for file_path in files:
        try:
            dataset.append(create_file_entry(file_path, directory_path))
        except FileNotFoundError as fnfe:
            print(f"[{fnfe}] File was deleted during processing: {file_path}")
            continue
        except PermissionError as pe:
            print(f"[{pe}] Permission denied: {file_path}")
            continue

    return dataset

In [ ]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean a dataframe.

    Args:
        df (pd.DataFrame): The dataframe to clean.

    Returns:
        pd.DataFrame: The cleaned dataframe.
    """

    df.columns = clean_headers(df.columns.to_list())

    relative_time: pd.Series | None = (
        df["Relative Time (s)"] if "Relative Time (s)" in df.columns else None
    )

    for metric in df.columns:
        if metric == "Timestamp":
            df["Timestamp"] = clean_timestamp(df["Timestamp"], relative_time)
            continue

        df[metric] = clean_data(df[metric])

    return df

In [ ]:
def load_dataframe(file: File) -> pd.DataFrame:
    """
    Load a dataframe from a file.

    Args:
        file (pd.Series): The file to load.

    Returns:
        pd.DataFrame: The loaded dataframe.
    """

    content: str = file.get("content", "")
    if not content or not content.strip():
        return pd.DataFrame()

    first_line: str = content.splitlines()[0]
    if not first_line:
        return pd.DataFrame()

    separator: str = detect_line_separator(first_line)

    extension: str = file.get("extension", "")
    filename: str = file.get("filename", "")

    if extension == EXTENSION_CSV:
        path: Path = file.get("path", Path())
        encoding: str = file.get("encoding", DEFAULT_ENCODING)

        if is_header(first_line):
            return pd.read_csv(path, encoding=encoding, sep=separator)
        else:
            return pd.read_csv(
                path,
                encoding=encoding,
                sep=separator,
                header=None,
                names=MISSING_HEADERS_DICT.get(filename, []),
            )

    elif extension == EXTENSION_HTML:
        return load_dataframe_from_html(content)

    elif extension == EXTENSION_LOG:
        lines: list[str] = content.splitlines()

        if is_header(first_line):
            return pd.DataFrame(lines[1:], columns=first_line.split(separator))
        else:
            return pd.DataFrame(lines, columns=MISSING_HEADERS_DICT.get(filename, []))

    return pd.DataFrame()

## Execution

### Exploration

In [ ]:
file_dataset: list[File] = build_dataset(RAW_DATA_DIR)

In [ ]:
df_dataset: pd.DataFrame = pd.DataFrame(file_dataset)
df_dataset

### Cleaning & Feature Engineering

In [ ]:
for file in tqdm(file_dataset, desc="Cleaning and feature engineering files"):
    df: pd.DataFrame = load_dataframe(file)

    df = clean_dataframe(df)

    output_filename: Path = file.get("relative_path", Path()).with_suffix(".csv")
    save_dataframe(df, CLEANED_DATA_DIR / output_filename)

    df = feature_dataframe(df, file.get("filename", ""))

    save_dataframe(df, FEATURED_DATA_DIR / output_filename)